# BÀI THỰC HÀNH 2: MẠNG NEURAL TÍCH CHẬP (CNN)\n\n[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Alice0602/DS201_Lab2/blob/main/assignments.ipynb)\n

<b>Hướng dẫn nộp bài:</b> Các bạn commit và push code lên github, sử dụng file txt đặt tên theo cú pháp <MSSV>.txt chứa đường link dẫn đến github của bài thực hành và nộp file txt này tên courses..

Bộ dữ liệu sử dụng: [VinaFood21 dataset](https://arxiv.org/abs/2108.02929).

Link download: https://drive.google.com/file/d/1UpZOf0XlwvB4rKpyZ35iwTA8oWHqDBbR/view?usp=share_link.

## 0. Thiết lập Môi trường & Bộ dữ liệu VinaFood21
Cell này hỗ trợ tự động nhận diện môi trường (**Google Colab**, **Kaggle**, hoặc **Local Machine**). Khi chạy trên Colab/Kaggle, script sẽ tự động cài thư viện và tải dữ liệu từ Google Drive nếu chưa có.


In [ ]:
# 1. Cài đặt thư viện cần thiết khi chạy trên Colab / Kaggle
import os
import sys
import zipfile
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
IN_KAGGLE = 'kaggle_web_client' in sys.modules

print(f"Môi trường: {'Google Colab' if IN_COLAB else ('Kaggle' if IN_KAGGLE else 'Local Machine')}")

if IN_COLAB:
    !pip install -q transformers scikit-learn matplotlib seaborn gdown

# 2. Tự động tải và giải nén bộ dữ liệu VinaFood21
DATA_DIR = Path("VinaFood21")
if IN_COLAB and not (DATA_DIR / "train").is_dir():
    print("Đang tải dữ liệu VinaFood21 từ Google Drive...")
    !gdown --id 1UpZOf0XlwvB4rKpyZ35iwTA8oWHqDBbR -O VinaFood21.zip
    print("Đang giải nén VinaFood21.zip...")
    with zipfile.ZipFile("VinaFood21.zip", "r") as z:
        z.extractall(".")
    print("Giải nén hoàn tất!")
elif IN_KAGGLE:
    kaggle_input = Path("/kaggle/input/vinafood21")
    if (kaggle_input / "train").is_dir():
        DATA_DIR = kaggle_input
    elif (Path("/kaggle/working/VinaFood21") / "train").is_dir():
        DATA_DIR = Path("/kaggle/working/VinaFood21")

print(f"Đường dẫn dữ liệu: {DATA_DIR.resolve()} | Tồn tại: {(DATA_DIR / 'train').is_dir()}")


Môi trường: Google Colab
Đang tải dữ liệu VinaFood21 từ Google Drive...
/usr/local/lib/python3.13/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=1UpZOf0XlwvB4rKpyZ35iwTA8oWHqDBbR
From (redirected): https://drive.google.com/uc?id=1UpZOf0XlwvB4rKpyZ35iwTA8oWHqDBbR&confirm=t&uuid=1a3e8c2c-44f6-43ad-9d8f-ae9c972010ba
To: /content/VinaFood21.zip
 93% 2.53G/2.71G [00:45<00:02, 67.4MB/s]

In [ ]:
# Khai báo thư viện, cố định seed và định nghĩa Data Pipeline
import random
import time
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch Version :", torch.__version__)
print("Thiết bị sử dụng:", device)

# Chuẩn hoá theo ImageNet (mean và std chuẩn)
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

def build_transforms(img_size, augment=True):
    """Dựng transform: train có augment (nếu bật), val/test không augment"""
    if augment:
        train_tf = transforms.Compose([
            transforms.RandomResizedCrop(img_size, scale=(0.7, 1.0)),
            transforms.RandomHorizontalFlip(),
            transforms.RandomRotation(15),
            transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
            transforms.ToTensor(),
            transforms.Normalize(MEAN, STD),
        ])
    else:
        train_tf = transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(),
            transforms.Normalize(MEAN, STD),
        ])

    eval_tf = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize(MEAN, STD),
    ])
    return train_tf, eval_tf

def build_loaders(data_root="VinaFood21", img_size=64, batch_size=32, augment=True, val_ratio=0.1, num_workers=2, seed=42):
    """Tạo train, val, test DataLoader và danh sách classes"""
    root = Path(data_root)
    train_tf, eval_tf = build_transforms(img_size, augment=augment)

    if (root / "val").is_dir():
        train_ds = datasets.ImageFolder(root / "train", transform=train_tf)
        val_ds   = datasets.ImageFolder(root / "val", transform=eval_tf)
        classes  = train_ds.classes
    else:
        full_train = datasets.ImageFolder(root / "train", transform=train_tf)
        full_eval  = datasets.ImageFolder(root / "train", transform=eval_tf)
        g = torch.Generator().manual_seed(seed)
        idx = torch.randperm(len(full_train), generator=g).tolist()
        n_val = int(len(idx) * val_ratio)
        train_ds = Subset(full_train, idx[n_val:])
        val_ds   = Subset(full_eval, idx[:n_val])
        classes  = full_train.classes

    test_ds = datasets.ImageFolder(root / "test", transform=eval_tf)

    train_dl = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=num_workers)
    val_dl   = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers)
    test_dl  = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers)

    return train_dl, val_dl, test_dl, classes


In [ ]:
# Khung huấn luyện, đánh giá và vẽ đồ thị dùng chung cho cả 4 bài
def n_params(module, trainable_only=False):
    """Đếm tổng số tham số hoặc chỉ tham số có requires_grad=True"""
    ps = module.parameters()
    if trainable_only:
        ps = (p for p in module.parameters() if p.requires_grad)
    return sum(p.numel() for p in ps)

def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total, correct, running = 0, 0, 0.0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()

        running += loss.item() * yb.size(0)
        pred = logits.argmax(dim=1)
        correct += (pred == yb).sum().item()
        total += yb.size(0)
    return running / total, correct / total

@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total, correct, running = 0, 0, 0.0
    y_true, y_pred = [], []
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)
        running += criterion(logits, yb).item() * yb.size(0)
        pred = logits.argmax(dim=1)
        correct += (pred == yb).sum().item()
        total += yb.size(0)
        y_true += yb.cpu().tolist()
        y_pred += pred.cpu().tolist()
    return running / total, correct / total, y_true, y_pred

def plot_loss_curve(history, model_name="Mô hình"):
    """Vẽ biểu đồ train loss vs val loss ngay sau khi huấn luyện"""
    epochs = range(1, len(history["train_loss"]) + 1)
    best_ep = int(np.argmin(history["val_loss"])) + 1

    plt.figure(figsize=(7, 3.8))
    plt.plot(epochs, history["train_loss"], "o-", label="Train Loss")
    plt.plot(epochs, history["val_loss"], "s-", label="Val Loss")
    plt.axvline(best_ep, ls="--", color="gray", label=f"Val Loss thấp nhất (Epoch {best_ep})")
    plt.xlabel("Epoch")
    plt.ylabel("Cross-Entropy Loss")
    plt.title(f"Đường cong Loss - {model_name}")
    plt.xticks(list(epochs)[::2] if len(epochs) > 5 else list(epochs))
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()
    print(f"-> Epoch có val_loss thấp nhất: {best_ep} | Val Loss = {min(history['val_loss']):.4f}")

def train_model(model, train_dl, val_dl, epochs=20, lr=1e-3, model_name="Mô hình", save_best=True):
    """Vòng lặp huấn luyện chuẩn, tối ưu bằng Adam và lưu checkpoint tốt nhất"""
    model.to(device)
    criterion = nn.CrossEntropyLoss()
    trainable_params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.Adam(trainable_params, lr=lr)

    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": [], "val_f1": []}
    best_f1 = -1.0
    best_state = None
    total_train_time = 0.0

    print(f"\n===== Bắt đầu huấn luyện: {model_name} =====")
    print(f"Epochs: {epochs} | LR: {lr} | Tham số cần học: {n_params(model, trainable_only=True):,}")
    for ep in range(1, epochs + 1):
        t0 = time.time()
        tr_loss, tr_acc = train_one_epoch(model, train_dl, criterion, optimizer, device)
        va_loss, va_acc, yt, yp = evaluate(model, val_dl, criterion, device)
        va_f1 = precision_recall_fscore_support(yt, yp, average="macro", zero_division=0)[2]
        elapsed = time.time() - t0
        total_train_time += elapsed

        history["train_loss"].append(tr_loss)
        history["train_acc"].append(tr_acc)
        history["val_loss"].append(va_loss)
        history["val_acc"].append(va_acc)
        history["val_f1"].append(va_f1)

        mark = ""
        if va_f1 > best_f1:
            best_f1 = va_f1
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            mark = " *"

        print(f"Epoch {ep:>2}/{epochs} | Train Loss: {tr_loss:.4f} Acc: {tr_acc:.4f} | Val Loss: {va_loss:.4f} Acc: {va_acc:.4f} F1: {va_f1:.4f} | {elapsed:.1f}s{mark}")

    if save_best and best_state is not None:
        model.load_state_dict(best_state)
        print(f"Đã phục hồi trọng số tốt nhất (val macro-F1: {best_f1:.4f})")

    plot_loss_curve(history, model_name=model_name)
    sec_per_epoch = total_train_time / epochs
    return history, sec_per_epoch

def evaluate_test_set(model, test_dl, classes, model_name="Mô hình"):
    """Đánh giá 4 độ đo (Acc, Precision, Recall, F1-macro) và vẽ ma trận nhầm lẫn 21x21"""
    criterion = nn.CrossEntropyLoss()
    test_loss, test_acc, y_true, y_pred = evaluate(model, test_dl, criterion, device)
    p_macro, r_macro, f1_macro, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)

    print(f"\n==================== KẾT QUẢ ĐÁNH GIÁ: {model_name} ====================")
    print(f"Test Loss        : {test_loss:.4f}")
    print(f"Accuracy         : {test_acc:.4f}")
    print(f"Precision (macro): {p_macro:.4f}")
    print(f"Recall (macro)   : {r_macro:.4f}")
    print(f"F1-score (macro) : {f1_macro:.4f}")
    print("-" * 72)
    print("Báo cáo chi tiết từng lớp món ăn:")
    print(classification_report(y_true, y_pred, target_names=classes, digits=4, zero_division=0))

    # Vẽ ma trận nhầm lẫn
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=False, fmt="d", cmap="Blues", xticklabels=classes, yticklabels=classes)
    plt.xlabel("Nhãn dự đoán")
    plt.ylabel("Nhãn thật")
    plt.title(f"Ma trận nhầm lẫn 21x21 - {model_name}")
    plt.xticks(rotation=90)
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()

    return {"acc": test_acc, "p": p_macro, "r": r_macro, "f1": f1_macro}


### Bài 1: Xây dựng mô hình LeNet. Huấn luyện và đánh giá mô hình LeNet trên 4 độ đo precision, recall và F1-macro (sử dụng Adam làm optimizer).

![image.png](attachment:image.png)

In [ ]:
# 1. Định nghĩa LeNet-5 điều chỉnh cho ảnh màu 64x64 và 21 lớp
class LeNet5(nn.Module):
    def __init__(self, in_ch=3, img=64, num_classes=21, pool="avg"):
        super().__init__()
        Pool = nn.AvgPool2d if pool == "avg" else nn.MaxPool2d
        self.c1 = nn.Conv2d(in_ch, 6, kernel_size=5)
        self.s2 = Pool(2, 2)
        self.c3 = nn.Conv2d(6, 16, kernel_size=5)
        self.s4 = Pool(2, 2)

        n_flat = self._flat_dim(in_ch, img)
        self.f5 = nn.Linear(n_flat, 120)
        self.f6 = nn.Linear(120, 84)
        self.out = nn.Linear(84, num_classes)
        self.n_flat = n_flat

    def _flat_dim(self, in_ch, img):
        # Dò số chiều sau S4 bằng một dummy forward
        with torch.no_grad():
            z = torch.zeros(1, in_ch, img, img)
            z = self.s4(self.c3(self.s2(self.c1(z))))
        return int(z.flatten(1).shape[1])

    def forward(self, x):
        x = self.s2(torch.relu(self.c1(x)))
        x = self.s4(torch.relu(self.c3(x)))
        x = x.flatten(1)
        x = torch.relu(self.f5(x))
        x = torch.relu(self.f6(x))
        return self.out(x)

# Kiểm tra số tham số
lenet_avg = LeNet5(in_ch=3, img=64, num_classes=21, pool="avg")
print("Số chiều sau flatten:", lenet_avg.n_flat)
print("Tổng tham số LeNet-5:", f"{n_params(lenet_avg):,}")
assert lenet_avg.n_flat == 2704, "Số chiều flatten phải là 2704"
assert n_params(lenet_avg) == 339_421, "Tổng tham số LeNet-5 phải là 339,421"
print("Xác nhận kiến trúc và tham số LeNet-5 hoàn toàn khớp với đề bài!")


In [ ]:
# 2. Huấn luyện và đánh giá LeNet-5 (AvgPool)
train_dl_64, val_dl_64, test_dl_64, classes = build_loaders(
    data_root=DATA_DIR, img_size=64, batch_size=32, augment=True
)

set_seed(42)
lenet_avg = LeNet5(in_ch=3, img=64, num_classes=len(classes), pool="avg")
hist_lenet_avg, sec_lenet_avg = train_model(
    lenet_avg, train_dl_64, val_dl_64, epochs=20, lr=1e-3, model_name="LeNet-5 (AvgPool)"
)

metrics_lenet_avg = evaluate_test_set(lenet_avg, test_dl_64, classes, model_name="LeNet-5 (AvgPool)")


In [ ]:
# 3. Thử nghiệm so sánh: LeNet-5 với MaxPool2d
set_seed(42)
lenet_max = LeNet5(in_ch=3, img=64, num_classes=len(classes), pool="max")
hist_lenet_max, sec_lenet_max = train_model(
    lenet_max, train_dl_64, val_dl_64, epochs=20, lr=1e-3, model_name="LeNet-5 (MaxPool)"
)

metrics_lenet_max = evaluate_test_set(lenet_max, test_dl_64, classes, model_name="LeNet-5 (MaxPool)")

print("\n===== SO SÁNH AVGPOOL VS MAXPOOL TRÊN LENET-5 =====")
print(f"LeNet-5 (AvgPool) Test Macro-F1: {metrics_lenet_avg['f1']:.4f}")
print(f"LeNet-5 (MaxPool) Test Macro-F1: {metrics_lenet_max['f1']:.4f}")


### Bài 2: Xây dựng mô hình GoogLeNet. Huấn luyện và đánh giá mô hình GoogLeNet trên 4 độ đo precision, recall và F1 (sử dụng Adam làm optimizer).

**Inception Blocks**

![image.png](attachment:image.png)

**GoogLeNet (Multi-branch Network)**

![image-2.png](attachment:image-2.png)

In [ ]:
# 1. Định nghĩa Inception Block và GoogLeNet rút gọn (3 Inception Blocks)
class InceptionBlock(nn.Module):
    def __init__(self, in_ch, f1, f3r, f3, f5r, f5, fp):
        super().__init__()
        # Nhánh 1: Conv 1x1
        self.b1 = nn.Sequential(
            nn.Conv2d(in_ch, f1, 1), nn.ReLU())
        # Nhánh 2: Conv 1x1 (bottleneck) -> Conv 3x3
        self.b2 = nn.Sequential(
            nn.Conv2d(in_ch, f3r, 1), nn.ReLU(),
            nn.Conv2d(f3r, f3, 3, padding=1), nn.ReLU())
        # Nhánh 3: Conv 1x1 (bottleneck) -> Conv 5x5
        self.b3 = nn.Sequential(
            nn.Conv2d(in_ch, f5r, 1), nn.ReLU(),
            nn.Conv2d(f5r, f5, 5, padding=2), nn.ReLU())
        # Nhánh 4: MaxPool 3x3 -> Conv 1x1
        self.b4 = nn.Sequential(
            nn.MaxPool2d(3, stride=1, padding=1),
            nn.Conv2d(in_ch, fp, 1), nn.ReLU())

    def forward(self, x):
        # Nối các nhánh theo chiều channel (dim=1)
        return torch.cat([self.b1(x), self.b2(x), self.b3(x), self.b4(x)], dim=1)

class MiniGoogLeNet(nn.Module):
    def __init__(self, in_ch=3, num_classes=21):
        super().__init__()
        # Stem conv: đưa ảnh 64x64 vào 64 channels
        self.stem = nn.Sequential(
            nn.Conv2d(in_ch, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)  # [B, 64, 32, 32]
        )
        # Block 1: in_ch=64 -> out_ch = 32 + 48 + 16 + 16 = 112
        self.incept1 = InceptionBlock(64, f1=32, f3r=32, f3=48, f5r=8, f5=16, fp=16)

        # Block 2: in_ch=112 -> out_ch = 64 + 96 + 32 + 32 = 224
        self.incept2 = InceptionBlock(112, f1=64, f3r=64, f3=96, f5r=16, f5=32, fp=32)

        self.pool = nn.MaxPool2d(2, 2)  # [B, 224, 16, 16]

        # Block 3: in_ch=224 -> out_ch = 128 + 160 + 32 + 32 = 352
        self.incept3 = InceptionBlock(224, f1=128, f3r=96, f3=160, f5r=16, f5=32, fp=32)

        # Global Average Pooling + Phân loại
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.dropout = nn.Dropout(0.4)
        self.fc = nn.Linear(352, num_classes)

    def forward(self, x):
        x = self.stem(x)
        x = self.incept1(x)
        x = self.incept2(x)
        x = self.pool(x)
        x = self.incept3(x)
        x = self.gap(x).flatten(1)
        x = self.dropout(x)
        return self.fc(x)

# In số channel và tham số của từng Inception Block
googlenet = MiniGoogLeNet(in_ch=3, num_classes=21)
dummy_x = torch.rand(1, 3, 64, 64)
with torch.no_grad():
    s = googlenet.stem(dummy_x)
    i1 = googlenet.incept1(s)
    i2 = googlenet.incept2(i1)
    p = googlenet.pool(i2)
    i3 = googlenet.incept3(p)

print("===== KIỂM TRA SHAPE & CHANNEL CÁC INCEPTION BLOCKS =====")
print(f"Stem output shape       : {tuple(s.shape)}")
print(f"Inception 1 output shape: {tuple(i1.shape)} | Channels: {i1.shape[1]} | Tham số: {n_params(googlenet.incept1):,}")
print(f"Inception 2 output shape: {tuple(i2.shape)} | Channels: {i2.shape[1]} | Tham số: {n_params(googlenet.incept2):,}")
print(f"Inception 3 output shape: {tuple(i3.shape)} | Channels: {i3.shape[1]} | Tham số: {n_params(googlenet.incept3):,}")
print(f"Tổng tham số GoogLeNet  : {n_params(googlenet):,}")


In [ ]:
# 2. Huấn luyện và đánh giá GoogLeNet
set_seed(42)
googlenet = MiniGoogLeNet(in_ch=3, num_classes=len(classes))
hist_googlenet, sec_googlenet = train_model(
    googlenet, train_dl_64, val_dl_64, epochs=20, lr=1e-3, model_name="GoogLeNet"
)

metrics_googlenet = evaluate_test_set(googlenet, test_dl_64, classes, model_name="GoogLeNet")


### Bài 3*: Xây dựng mô hình ResNet-18, đánh giá mô hình ResNet-18 trên bộ dữ liệu VinaFood21 sử dụng các độ đo precision, recall, và F1 (Sử dụng Adam làm optimizer).

**ResNet Block**

![image.png](attachment:image.png)

**ResNet**

![image-2.png](attachment:image-2.png)

In [ ]:
# 1. Định nghĩa BasicBlock và ResNet-18 [2, 2, 2, 2] từ đầu (from scratch)
class BasicBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, 1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_ch)

        # Shortcut connection
        self.shortcut = nn.Identity()
        if stride != 1 or in_ch != out_ch:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 1, stride, bias=False),
                nn.BatchNorm2d(out_ch)
            )

    def forward(self, x):
        out = torch.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out = out + self.shortcut(x)  # Phép cộng residual trước
        return torch.relu(out)        # Kích hoạt ReLU sau

class ResNet18(nn.Module):
    def __init__(self, num_classes=21):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, 64, 7, stride=2, padding=3, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(3, stride=2, padding=1)
        )
        self.layer1 = self._stage(64, 64, stride=1)
        self.layer2 = self._stage(64, 128, stride=2)
        self.layer3 = self._stage(128, 256, stride=2)
        self.layer4 = self._stage(256, 512, stride=2)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(512, num_classes)

    @staticmethod
    def _stage(in_ch, out_ch, stride):
        return nn.Sequential(
            BasicBlock(in_ch, out_ch, stride),
            BasicBlock(out_ch, out_ch, 1)
        )

    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.pool(x).flatten(1)
        return self.fc(x)

# Kiểm tra số tham số ResNet-18 (phải khớp chính xác 11,187,285 tham số)
r18_check = ResNet18(num_classes=21)
print("Tổng tham số ResNet-18:", f"{n_params(r18_check):,}")
assert n_params(r18_check) == 11_187_285, "Số tham số ResNet-18 phải là 11,187,285"
print("Xác nhận ResNet-18 chuẩn xác hoàn toàn theo giáo trình!")


In [ ]:
# 2. Dataloader cho ResNet-18 (ảnh 96x96) và Huấn luyện LẦN 1: KHÔNG CÓ Augmentation
IMG_SIZE_R18 = 96

# DataLoader KHÔNG có Augmentation
train_dl_r18_noaug, val_dl_r18_noaug, test_dl_r18, _ = build_loaders(
    data_root=DATA_DIR, img_size=IMG_SIZE_R18, batch_size=32, augment=False
)

set_seed(42)
r18_noaug = ResNet18(num_classes=len(classes))
hist_r18_noaug, sec_r18_noaug = train_model(
    r18_noaug, train_dl_r18_noaug, val_dl_r18_noaug, epochs=20, lr=1e-3,
    model_name="ResNet-18 (KHÔNG Augment)"
)

metrics_r18_noaug = evaluate_test_set(r18_noaug, test_dl_r18, classes, model_name="ResNet-18 (KHÔNG Augment)")


In [ ]:
# 3. Huấn luyện LẦN 2: CÓ Augmentation
# DataLoader CÓ Augmentation
train_dl_r18_aug, val_dl_r18_aug, _, _ = build_loaders(
    data_root=DATA_DIR, img_size=IMG_SIZE_R18, batch_size=32, augment=True
)

set_seed(42)
r18_aug = ResNet18(num_classes=len(classes))
hist_r18_aug, sec_r18_aug = train_model(
    r18_aug, train_dl_r18_aug, val_dl_r18_aug, epochs=20, lr=1e-3,
    model_name="ResNet-18 (CÓ Augment)"
)

metrics_r18_aug = evaluate_test_set(r18_aug, test_dl_r18, classes, model_name="ResNet-18 (CÓ Augment)")


In [ ]:
# 4. So sánh đường Loss & Phân tích hiện tượng Overfitting của ResNet-18
epochs_r18 = range(1, len(hist_r18_noaug["train_loss"]) + 1)

plt.figure(figsize=(12, 4.5))

plt.subplot(1, 2, 1)
plt.plot(epochs_r18, hist_r18_noaug["train_loss"], "o--", label="Train Loss (Không Augment)")
plt.plot(epochs_r18, hist_r18_noaug["val_loss"], "s-", label="Val Loss (Không Augment)")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Lần 1: Không có Augmentation (Bị Overfitting)")
plt.legend()
plt.grid(alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(epochs_r18, hist_r18_aug["train_loss"], "o--", label="Train Loss (Có Augment)", color="green")
plt.plot(epochs_r18, hist_r18_aug["val_loss"], "s-", label="Val Loss (Có Augment)", color="red")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Lần 2: Có Augmentation (Giảm Overfitting)")
plt.legend()
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("\n===== SO SÁNH RESNET-18: CÓ VS KHÔNG AUGMENTATION =====")
print(f"KHÔNG Augmentation -> Test Macro-F1: {metrics_r18_noaug['f1']:.4f}")
print(f"CÓ Augmentation    -> Test Macro-F1: {metrics_r18_aug['f1']:.4f}")
print(f"Chênh lệch Macro-F1 : {metrics_r18_aug['f1'] - metrics_r18_noaug['f1']:+.4f}")
print("""
Nhận xét:
1. Khi KHÔNG có Augmentation: ResNet-18 có tới 11.18 triệu tham số trong khi tập dữ liệu nhỏ (~14k ảnh),
   dẫn đến đường Train Loss giảm mạnh về gần 0 nhưng Val Loss lại bật tăng (Overfitting nghiêm trọng).
2. Khi CÓ Augmentation: Các phép biến đổi xoay, lật, cắt ngẫu nhiên tạo thêm dữ liệu đa dạng, giúp mô hình
   học các đặc trưng khái quát hóa tốt hơn, kìm hãm Val Loss và tăng Macro-F1 rõ rệt.
""")


### Bài 4*: Sử dụng pretrained ResNet50 từ HuggingFace để fine-tune trên bộ dữ liệu VinaFood21.

In [ ]:
# 1. Định nghĩa PretrainedResnet sử dụng microsoft/resnet-50 từ HuggingFace
from transformers import ResNetForImageClassification

class PretrainedResnet(nn.Module):
    def __init__(self, num_classes=21, freeze_backbone=False):
        super().__init__()
        base = ResNetForImageClassification.from_pretrained("microsoft/resnet-50")
        self.resnet = base.resnet
        self.classifier = nn.Linear(2048, num_classes, bias=True)
        if freeze_backbone:
            # Đóng băng backbone cho chế độ Feature Extraction
            for p in self.resnet.parameters():
                p.requires_grad = False

    def forward(self, images):
        features = self.resnet(images).pooler_output  # [B, 2048, 1, 1]
        features = features.squeeze(-1).squeeze(-1)   # [B, 2048]
        return self.classifier(features)              # [B, 21]

# Kiểm tra số tham số trong 2 chế độ
fe_test = PretrainedResnet(21, freeze_backbone=True)
ft_test = PretrainedResnet(21, freeze_backbone=False)

print(f"{'Chế độ':<22}{'Tổng tham số':>16}{'Cần cập nhật':>16}{'Learning Rate':>14}")
print(f"{'Feature Extraction':<22}{n_params(fe_test):>16,}{n_params(fe_test, True):>16,}{'1e-3':>14}")
print(f"{'Fine-Tuning':<22}{n_params(ft_test):>16,}{n_params(ft_test, True):>16,}{'1e-4':>14}")

assert n_params(fe_test) == 23_551_061
assert n_params(fe_test, trainable_only=True) == 43_029, "Feature Extraction chỉ cập nhật 43,029 tham số của classifier"
assert n_params(ft_test, trainable_only=True) == 23_551_061, "Fine-tuning cập nhật toàn bộ 23.5M tham số"
print("Kiểm tra tham số ResNet-50 hoàn tất!")


In [ ]:
# 2. Huấn luyện ResNet-50 Pretrained (ảnh 224x224)
IMG_SIZE_R50 = 224

train_dl_224, val_dl_224, test_dl_224, _ = build_loaders(
    data_root=DATA_DIR, img_size=IMG_SIZE_R50, batch_size=32, augment=True
)

# Chế độ 1: Feature Extraction (đóng băng backbone, Adam lr=1e-3)
print("\n==================== CHẾ ĐỘ 1: FEATURE EXTRACTION ====================")
set_seed(42)
r50_fe = PretrainedResnet(num_classes=len(classes), freeze_backbone=True)
hist_r50_fe, sec_r50_fe = train_model(
    r50_fe, train_dl_224, val_dl_224, epochs=15, lr=1e-3, model_name="ResNet-50 (Feature Extraction)"
)
metrics_r50_fe = evaluate_test_set(r50_fe, test_dl_224, classes, model_name="ResNet-50 (Feature Extraction)")

# Chế độ 2: Fine-Tuning (mở toàn bộ mạng, Adam lr=1e-4)
print("\n==================== CHẾ ĐỘ 2: FINE-TUNING ====================")
set_seed(42)
r50_ft = PretrainedResnet(num_classes=len(classes), freeze_backbone=False)
hist_r50_ft, sec_r50_ft = train_model(
    r50_ft, train_dl_224, val_dl_224, epochs=10, lr=1e-4, model_name="ResNet-50 (Fine-Tuning)"
)
metrics_r50_ft = evaluate_test_set(r50_ft, test_dl_224, classes, model_name="ResNet-50 (Fine-Tuning)")


## Bảng Tổng hợp Kết quả Thực nghiệm & Hướng dẫn Nộp bài


In [ ]:
# Bảng tổng hợp kết quả của cả 4 bài thực hành
import pandas as pd

summary_data = [
    {
        "Mô hình": "LeNet-5 (AvgPool)",
        "Resize": "64x64",
        "Số tham số": f"{n_params(lenet_avg):,}",
        "Giây/epoch": f"{sec_lenet_avg:.1f}s",
        "Accuracy": f"{metrics_lenet_avg['acc']:.4f}",
        "Macro-F1": f"{metrics_lenet_avg['f1']:.4f}"
    },
    {
        "Mô hình": "LeNet-5 (MaxPool)",
        "Resize": "64x64",
        "Số tham số": f"{n_params(lenet_max):,}",
        "Giây/epoch": f"{sec_lenet_max:.1f}s",
        "Accuracy": f"{metrics_lenet_max['acc']:.4f}",
        "Macro-F1": f"{metrics_lenet_max['f1']:.4f}"
    },
    {
        "Mô hình": "GoogLeNet (3 Inception)",
        "Resize": "64x64",
        "Số tham số": f"{n_params(googlenet):,}",
        "Giây/epoch": f"{sec_googlenet:.1f}s",
        "Accuracy": f"{metrics_googlenet['acc']:.4f}",
        "Macro-F1": f"{metrics_googlenet['f1']:.4f}"
    },
    {
        "Mô hình": "ResNet-18 (from scratch, No Aug)*",
        "Resize": "96x96",
        "Số tham số": f"{n_params(r18_noaug):,}",
        "Giây/epoch": f"{sec_r18_noaug:.1f}s",
        "Accuracy": f"{metrics_r18_noaug['acc']:.4f}",
        "Macro-F1": f"{metrics_r18_noaug['f1']:.4f}"
    },
    {
        "Mô hình": "ResNet-18 (from scratch, Aug)*",
        "Resize": "96x96",
        "Số tham số": f"{n_params(r18_aug):,}",
        "Giây/epoch": f"{sec_r18_aug:.1f}s",
        "Accuracy": f"{metrics_r18_aug['acc']:.4f}",
        "Macro-F1": f"{metrics_r18_aug['f1']:.4f}"
    },
    {
        "Mô hình": "ResNet-50 (Feature Extraction)*",
        "Resize": "224x224",
        "Số tham số": f"{n_params(r50_fe):,} ({n_params(r50_fe, True):,} train)",
        "Giây/epoch": f"{sec_r50_fe:.1f}s",
        "Accuracy": f"{metrics_r50_fe['acc']:.4f}",
        "Macro-F1": f"{metrics_r50_fe['f1']:.4f}"
    },
    {
        "Mô hình": "ResNet-50 (Fine-Tuning)*",
        "Resize": "224x224",
        "Số tham số": f"{n_params(r50_ft):,}",
        "Giây/epoch": f"{sec_r50_ft:.1f}s",
        "Accuracy": f"{metrics_r50_ft['acc']:.4f}",
        "Macro-F1": f"{metrics_r50_ft['f1']:.4f}"
    }
]

df_summary = pd.DataFrame(summary_data)
print("\n" + "="*80)
print("BẢNG TỔNG HỢP KẾT QUẢ THỰC NGHIỆM")
print("="*80)
print(df_summary.to_string(index=False))


### Hướng dẫn Commit, Push lên GitHub và Nộp bài

Sau khi hoàn tất chạy notebook trên Colab hoặc máy cá nhân, bạn thực hiện các bước sau để nộp bài:

1. **Lưu notebook có đầy đủ kết quả (Outputs, biểu đồ loss, ma trận nhầm lẫn):**
   - Trên Colab: Vào **File** -> **Save a copy in GitHub** (chọn repository của bạn), hoặc tải file `.ipynb` về máy rồi commit vào repository.
2. **Commit và Push lên GitHub:**
   ```bash
   git add assignments.ipynb
   git commit -m "Complete Lab 2: CNN with LeNet, GoogLeNet, ResNet-18, and ResNet-50"
   git push origin main
   ```
3. **Tạo file nộp bài:**
   - Tạo file text đặt tên là `<MSSV>.txt` (ví dụ `21120000.txt`).
   - Ghi đường link dẫn đến file notebook trên GitHub của bạn vào bên trong file txt (ví dụ: `https://github.com/<username>/<repo>/blob/main/assignments.ipynb`).
   - Nộp file `<MSSV>.txt` này lên hệ thống khóa học (Courses).
